In [ ]:
# # Make sure the repository root is on sys.path so `quantum_inferno` can be imported when running this notebook from `docs/notebooks/`
# import os, sys
# from pathlib import Path
# repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
# if repo_root not in sys.path:
#     sys.path.insert(0, repo_root)

# Imports
import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy.signal import get_window, windows
from quantum_inferno.utilities.window import taper_power_correction

## Example parameters and helper function
Define the signal, sample rates, and a helper to obtain SciPy windows (fixing a minor naming bug from the script).

In [ ]:
# Window types to compare
window_types = ['boxcar', 'gaussian', 'hann', 'tukey', 'blackman']

frequency_sample_rate_hz = 800.
frequency_design_center_hz = 60.
sig_amplitude = 1.
sig_duration_s = 60.
dft_samples = int(sig_duration_s * frequency_sample_rate_hz)
time_cyber = np.arange(dft_samples)
gauss_std = (dft_samples - 1) / np.pi / 2

def get_scipy_window(win_name: str, num_samples: int, fftbins: bool = True):
    if win_name == 'gaussian':
        return get_window((win_name, gauss_std), num_samples, fftbins=fftbins)
    else:
        return get_window(win_name, num_samples, fftbins=fftbins)

frequency_center = frequency_design_center_hz / frequency_sample_rate_hz
sig_cosine = sig_amplitude * np.cos(2 * np.pi * frequency_center * time_cyber)
sig_var = np.var(sig_cosine)

# FFT design (choose fft length as power of two based on desired spectral resolution)
frequency_resolution_dft_hz = frequency_sample_rate_hz / dft_samples
fft_design_resolution_hz = 0.25 * frequency_resolution_dft_hz
fft_design_time_s = 1. / fft_design_resolution_hz
fft_samples = 2 ** (int(np.ceil(np.log2(fft_design_time_s * frequency_sample_rate_hz))))

frequency_resolution_fft_hz = frequency_sample_rate_hz / fft_samples

# Frequency arrays used later
frequency_dft_pos_hz = scipy.fft.rfftfreq(dft_samples, d=1/frequency_sample_rate_hz)
frequency_fft_pos_hz = scipy.fft.rfftfreq(fft_samples, d=1/frequency_sample_rate_hz)
frequency_fft_over_df = scipy.fft.rfftfreq(fft_samples, d=1/dft_samples)

print('DFT samples:', dft_samples)
print('FFT samples:', fft_samples)

## Visualize taper windows
Plot the time-domain tapers used in the example.

In [ ]:
fg1, ax1 = plt.subplots(len(window_types), 1, sharex='all', sharey='all', figsize=(8., 6.))
for c_, (w_name_, ax_) in enumerate(zip(window_types, ax1)):
    win_taper = get_scipy_window(w_name_, dft_samples, fftbins=False)
    ax_.plot(win_taper, f'C{c_}-', label=w_name_)
    ax_.text(0.02, 0.5, w_name_, color=f'C{c_}', verticalalignment='bottom',
             horizontalalignment='left', bbox={'color': 'white', 'pad': 0})
    ax_.grid()
ax1[0].set_title('Example Taper Windows')
fg1.tight_layout(h_pad=0.6)
plt.show()

## Spectral leakage of the windows
Compute and plot the (normalized) magnitude of the window FFTs to inspect sidelobes and leakage.

In [ ]:
fg0, axx = plt.subplots(len(window_types), 1, sharex='all', sharey='all', figsize=(8., 6.))
for c_, (w_name_, ax_) in enumerate(zip(window_types, axx)):
    win_taper = get_scipy_window(w_name_, dft_samples, fftbins=False)
    W_ = scipy.fft.rfft(win_taper / np.abs(np.sum(win_taper)), n=fft_samples)
    W_dB = 20 * np.log10(np.maximum(np.abs(W_), 1e-250))
    ax_.plot(frequency_fft_over_df, W_dB, f'C{c_}-', label=w_name_)
    ax_.text(0.1, -50, w_name_, color=f'C{c_}', verticalalignment='bottom',
             horizontalalignment='left', bbox={'color': 'white', 'pad': 0})
    ax_.set_yticks([-20, -60])
    ax_.grid()
axx[0].set_title('Spectral Leakage of Example Windows')
fg0.supylabel(r"Normalized Magnitude $20\,\log_{10}|W(f)/c^\operatorname{amp}|$ in dB", x=0.04, y=0.5, fontsize='medium')
axx[-1].set(xlabel=r"Normalized frequency $f/\Delta f$ in bins", xlim=(0, 9), ylim=(-75, 3))
fg0.tight_layout(h_pad=0.6)
plt.show()

## Apply tapers to the tone and inspect FFT power
For each taper, apply it to the tone, compute the FFT (zero-padded to `fft_samples`), apply the energy correction factor, and plot the power in dB.
The notebook prints per-window statistics (mean, rms, correction factors) as in the script.

In [ ]:
fg2, axx = plt.subplots(len(window_types), 1, sharex='all', sharey='all', figsize=(8., 8.))
for c_, (w_name_, ax_) in enumerate(zip(window_types, axx)):
    win_taper = get_scipy_window(w_name_, dft_samples, fftbins=False)
    sig_cosine_tapered = win_taper * sig_cosine
    sig_tapered_var = np.var(sig_cosine_tapered)

    win_mean = np.abs(np.sum(win_taper)) / dft_samples
    win_var = np.sum(win_taper ** 2) / dft_samples
    win_rms = np.sqrt(win_var)
    spectral_amplitude_correction_factor = 1 / win_mean
    spectral_energy_correction_factor = 1 / win_rms

    print('*** Taper window: ', w_name_)
    print(' Input signal variance: ', sig_var)
    print(' Tapered signal variance: ', sig_tapered_var)
    print(' Taper window mean amp: ', win_mean)
    print(' Taper window rms: ', win_rms)
    print(' Taper amplitude correction factor (ACF = 1/mean): ', spectral_amplitude_correction_factor)
    print(' Taper energy correction factor (ECF = 1/rms): ', spectral_energy_correction_factor)

    # FFT of tapered signal, zero-padded/truncated to fft_samples
    fft_sig_pos = scipy.fft.rfft(sig_cosine_tapered, n=fft_samples)
    fft_square = np.abs(fft_sig_pos) ** 2
    frequency_resolution_fft = 1. / fft_samples
    frequency_resolution_fft_hz = frequency_sample_rate_hz * frequency_resolution_fft
    fft_power = 2. * frequency_resolution_fft * fft_square / dft_samples

    # Apply energy correction factor (ECF) to compensate taper RMS loss
    W_dB = 10 * np.log10(np.maximum(fft_power * spectral_energy_correction_factor ** 2, 1e-250))
    ax_.plot(frequency_fft_pos_hz, W_dB, f'C{c_}-', label=w_name_)
    ax_.text(frequency_design_center_hz, -160, w_name_, color=f'C{c_}', verticalalignment='bottom',
             horizontalalignment='left', bbox={'color': 'white', 'pad': 0})
    ax_.set_yticks([-140, -80, -6])
    ax_.grid()

axx[0].set_title('Spectral Leakage of Example Windows')
fg2.supylabel(r"$10\,\log_{10}<Power>$ in dB", x=0.04, y=0.5, fontsize='medium')
axx[-1].set(xlabel=r"Frequency, Hz", xlim=(58, 62), ylim=(-140, -0))
fg2.tight_layout(h_pad=0.6)
plt.show()